In [2]:
!pip install torch --extra-index-url https://download.pytorch.org/whl/cpu
!pip install miditok --upgrade

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\melik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\melik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
from miditok import REMI, TokenizerConfig
from miditok.pytorch_data import DatasetJSON, DataCollator
from torch.utils.data import DataLoader
from pathlib import Path
from miditok.utils import split_files_for_training


In [4]:
tokenizer = REMI(params="../data/processed/chopin_tokenizer.json")
midi_dir = Path("../data/processed/chopin_midi").resolve()
dataset_chunks_dir = Path("../data/processed/chopin_chunks").resolve()
token_dir = Path("../data/processed/tokens").resolve()
midi_paths = [p.resolve() for p in midi_dir.glob("*.midi")]


split_files_for_training(
    files_paths=midi_paths,
    tokenizer=tokenizer,
    save_dir=dataset_chunks_dir,
    max_seq_len=1024,
)
chunk_midi_paths = list(dataset_chunks_dir.glob("**/*.mid*"))
tokenizer.tokenize_midi_dataset(
    files_paths=chunk_midi_paths,
    out_dir=token_dir,
)

dataset = DatasetJSON(
    files_paths=list(token_dir.glob("**/*.json")),
    max_seq_len=1024,
    bos_token_id=tokenizer["BOS_None"], # dosya başı
    eos_token_id = tokenizer["EOS_None"], # dosya sonu
)

collator = DataCollator(
    tokenizer.pad_token_id,
    pad_on_left= False,
    copy_inputs_as_labels=True, 
)
dataloader = DataLoader(dataset, batch_size =16,shuffle=True, collate_fn=collator )


C:\Users\melik\AppData\Local\Temp\ipykernel_16252\1074933407.py:15: UserWarning: miditok: The `tokenize_midi_dataset` method had been renamed `tokenize_dataset`. It is now depreciated and will be removed in future updates.
  tokenizer.tokenize_midi_dataset(
Tokenizing music files (processed/tokens): 100%|██████████| 21696/21696 [08:02<00:00, 44.93it/s]
